# Yesterday Multi-Source Publications Summary

Pull yesterday's records from Hacker News, EDGAR, Federal Register, Google News, StackExchange, Crossref, and OpenAlex, then rank the major publications/entities with local extractive summaries.

In [ ]:
import json
import re
import sys
from collections import Counter
from datetime import date, timedelta
from pathlib import Path


def _find_project_src(start: Path) -> Path:
    for base in [start, *start.parents]:
        candidate = base / "src" / "data_ingestion"
        if candidate.exists():
            return base / "src"

    two_up = start.parent.parent
    candidate = two_up / "src" / "data_ingestion"
    if candidate.exists():
        return two_up / "src"

    raise RuntimeError("Could not locate project src/data_ingestion from notebook cwd")


SRC_PATH = _find_project_src(Path.cwd().resolve())
src_path_str = str(SRC_PATH)
if src_path_str in sys.path:
    sys.path.remove(src_path_str)
sys.path.insert(0, src_path_str)


def _uses_local_data_ingestion() -> bool:
    module = sys.modules.get("data_ingestion")
    if module is None:
        return True
    module_file = str(getattr(module, "__file__", ""))
    return module_file.startswith(src_path_str)


if not _uses_local_data_ingestion():
    for module_name in list(sys.modules):
        if module_name == "data_ingestion" or module_name.startswith("data_ingestion."):
            del sys.modules[module_name]

import data_ingestion

print(f"Using data_ingestion from {Path(data_ingestion.__file__).resolve()}")

import pandas as pd

from data_ingestion.exceptions import QuotaExceededError
from data_ingestion.pipeline import stream_transformed_records

## Run Configuration

In [ ]:
YESTERDAY = date.today() - timedelta(days=1)
YESTERDAY_ISO = YESTERDAY.isoformat()

GOOGLE_NEWS_QUERY = "technology OR science OR business OR health"
STACKEXCHANGE_SITE = "stackoverflow"

SOURCE_PAGE_SETTINGS = {
    "hackernews": {
        "max_pages": None,
        "hits_per_page": 1000,
        "http": {"max_requests_per_session": 1000},
    },
    "edgar": {
        "max_pages": None,
        "per_page": 1000,
        "http": {"max_requests_per_session": 5000},
    },
    "federalregister": {
        "max_pages": None,
        "per_page": 1000,
        "http": {"max_requests_per_session": 1000},
    },
    "googlenews": {
        "max_pages": None,
        "page_size": 1000,
        "http": {"requests_per_second": 0.2, "max_requests_per_session": 200},
    },
    "stackexchange": {
        "max_pages": None,
        "page_size": 100,
        "http": {"max_requests_per_session": 250},
    },
    "crossref": {
        "max_pages": None,
        "rows": 1000,
        "http": {"max_requests_per_session": 5000},
    },
    "openalex": {
        "max_pages": None,
        "per_page": 200,
        "http": {"max_requests_per_session": 5000},
    },
}

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RECORDS_OUTPUT = OUTPUT_DIR / "yesterday_publications_records.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "yesterday_publications_summary.csv"

print(f"Analyzing records from {YESTERDAY_ISO}")
pd.DataFrame(
    [
        {
            "source": source,
            "max_pages": settings["max_pages"],
            "request_budget": settings.get("http", {}).get("max_requests_per_session"),
            "requests_per_second": settings.get("http", {}).get("requests_per_second", 1.0),
        }
        for source, settings in SOURCE_PAGE_SETTINGS.items()
    ]
)

In [ ]:
fetcher_specs = [
    {
        "source": "hackernews",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            "hn_item_type": "story",
            "use_date_sort": True,
            **SOURCE_PAGE_SETTINGS["hackernews"],
        },
    },
    {
        "source": "edgar",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            **SOURCE_PAGE_SETTINGS["edgar"],
        },
    },
    {
        "source": "federalregister",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            **SOURCE_PAGE_SETTINGS["federalregister"],
        },
    },
    {
        "source": "googlenews",
        "config": {
            "query": GOOGLE_NEWS_QUERY,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            **SOURCE_PAGE_SETTINGS["googlenews"],
        },
    },
    {
        "source": "stackexchange",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            "site": STACKEXCHANGE_SITE,
            "sort": "creation",
            **SOURCE_PAGE_SETTINGS["stackexchange"],
        },
    },
    {
        "source": "crossref",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "date_mode": "publication",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            **SOURCE_PAGE_SETTINGS["crossref"],
        },
    },
    {
        "source": "openalex",
        "config": {
            "query": None,
            "search_mode": "date_only",
            "start_date": YESTERDAY_ISO,
            "end_date": YESTERDAY_ISO,
            **SOURCE_PAGE_SETTINGS["openalex"],
        },
    },
]

transform_spec = {
    "transforms": [
        {"op": "require_fields", "fields": ["title", "url"]},
        {"op": "dedupe", "keys": ["source", "external_id", "url"]},
    ]
}

## Pull And Normalize

In [ ]:
rows = []
run_notes = []

for spec in fetcher_specs:
    source_name = spec["source"]
    source_start_count = len(rows)
    try:
        for source, record in stream_transformed_records(
            [spec],
            transform_spec=transform_spec,
            start_date=YESTERDAY_ISO,
            end_date=YESTERDAY_ISO,
        ):
            rows.append(
                {
                    "source": source,
                    "external_id": record.external_id,
                    "title": record.title,
                    "authors": record.authors,
                    "published_date": (
                        record.published_date.isoformat()
                        if record.published_date is not None
                        else None
                    ),
                    "url": record.url,
                    "abstract": record.abstract,
                    "full_text": record.full_text,
                    "full_text_url": record.full_text_url,
                    "topic": record.topic,
                    "record_type": record.record_type.value,
                    "raw_payload": record.raw_payload,
                }
            )
    except QuotaExceededError as exc:
        run_notes.append(
            {
                "source": source_name,
                "status": "quota_exhausted",
                "records_kept": len(rows) - source_start_count,
                "message": str(exc),
            }
        )
        continue
    except Exception as exc:
        run_notes.append(
            {
                "source": source_name,
                "status": "failed",
                "records_kept": len(rows) - source_start_count,
                "message": str(exc),
            }
        )
        continue

    run_notes.append(
        {
            "source": source_name,
            "status": "completed",
            "records_kept": len(rows) - source_start_count,
            "message": "",
        }
    )

columns = [
    "source",
    "external_id",
    "title",
    "authors",
    "published_date",
    "url",
    "abstract",
    "full_text",
    "full_text_url",
    "topic",
    "record_type",
    "raw_payload",
]
df = pd.DataFrame(rows, columns=columns)
run_notes_df = pd.DataFrame(run_notes)
print(f"Fetched {len(df)} records")
display(run_notes_df)
df.head(10)

## Publication And Summary Helpers

In [ ]:
STOPWORDS = {
    "about",
    "after",
    "again",
    "against",
    "all",
    "also",
    "and",
    "are",
    "because",
    "been",
    "being",
    "between",
    "but",
    "can",
    "could",
    "from",
    "has",
    "have",
    "into",
    "its",
    "more",
    "new",
    "not",
    "over",
    "than",
    "that",
    "the",
    "their",
    "then",
    "there",
    "these",
    "this",
    "through",
    "under",
    "using",
    "was",
    "were",
    "when",
    "where",
    "which",
    "while",
    "with",
    "would",
    "your",
}


def first_text(*values):
    for value in values:
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, list):
            for item in value:
                if isinstance(item, str) and item.strip():
                    return item.strip()
    return None


def openalex_publication(raw):
    for location_key in ("primary_location", "best_oa_location"):
        location = raw.get(location_key) or {}
        source = location.get("source") or {}
        name = first_text(
            source.get("display_name"), source.get("display_name_alternatives")
        )
        if name:
            return name
    host_venue = raw.get("host_venue") or {}
    return first_text(host_venue.get("display_name"))


def publication_for_row(row):
    raw = row.get("raw_payload") if isinstance(row.get("raw_payload"), dict) else {}
    source = row.get("source")

    if source == "crossref":
        return (
            first_text(raw.get("container-title"), raw.get("publisher")) or "Crossref"
        )
    if source == "openalex":
        return openalex_publication(raw) or "OpenAlex"
    if source == "googlenews":
        title = row.get("title") or ""
        if " - " in title:
            return title.rsplit(" - ", 1)[-1].strip() or "Google News"
        return "Google News"
    if source == "edgar":
        src = raw.get("_source") or {}
        return first_text(src.get("display_names")) or "SEC EDGAR"
    if source == "federalregister":
        agencies = raw.get("agencies") or []
        for agency in agencies:
            name = first_text(agency.get("raw_name"), agency.get("name"))
            if name:
                return name
        return "Federal Register"
    if source == "stackexchange":
        return STACKEXCHANGE_SITE
    if source == "hackernews":
        return "Hacker News"
    return source or "unknown"


def make_summary_text(row):
    parts = [row.get("title"), row.get("abstract"), row.get("full_text")]
    return " ".join(
        str(part).strip() for part in parts if isinstance(part, str) and part.strip()
    )


def tokenize(text):
    for token in re.findall(r"[a-z][a-z0-9_-]{2,}", str(text).lower()):
        if token not in STOPWORDS:
            yield token


def extractive_summary(texts, max_sentences=3):
    joined = " ".join(
        str(text) for text in texts if isinstance(text, str) and text.strip()
    )
    sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\\s+|\\n+", joined) if s.strip()
    ]
    if not sentences:
        return ""

    term_counts = Counter(tokenize(joined))
    scored = []
    for idx, sentence in enumerate(sentences):
        terms = list(tokenize(sentence))
        if not terms:
            continue
        score = sum(term_counts[t] for t in terms) / max(len(terms), 1)
        scored.append((score, idx, sentence))

    selected = sorted(scored, reverse=True)[:max_sentences]
    selected = sorted(selected, key=lambda item: item[1])
    return " ".join(sentence for _, _, sentence in selected)


if not df.empty:
    df["publication"] = df.apply(publication_for_row, axis=1)
    df["summary_text"] = df.apply(make_summary_text, axis=1)
else:
    df["publication"] = pd.Series(dtype="object")
    df["summary_text"] = pd.Series(dtype="object")

df[["source", "publication", "title", "published_date", "url"]].head(10)

## Source, Publication, And Topic Rankings

In [ ]:
records_by_source = (
    df["source"].value_counts().rename_axis("source").reset_index(name="records")
    if not df.empty
    else pd.DataFrame(columns=["source", "records"])
)
records_by_source

In [ ]:
top_publications = (
    df["publication"]
    .fillna("unknown")
    .value_counts()
    .head(25)
    .rename_axis("publication")
    .reset_index(name="records")
    if not df.empty
    else pd.DataFrame(columns=["publication", "records"])
)
top_publications

In [ ]:
topic_counts = (
    df["topic"]
    .fillna("unknown")
    .value_counts()
    .head(25)
    .rename_axis("topic")
    .reset_index(name="records")
    if not df.empty
    else pd.DataFrame(columns=["topic", "records"])
)

term_counts = Counter()
for text in df["summary_text"].dropna() if "summary_text" in df else []:
    term_counts.update(tokenize(text))

top_terms = pd.DataFrame(term_counts.most_common(25), columns=["term", "count"])
display(topic_counts)
top_terms

## Representative Titles And Extractive Summaries

In [ ]:
summary_rows = []
for publication in top_publications["publication"].head(15):
    group = df[df["publication"] == publication].copy()
    titles = [title for title in group["title"].dropna().head(5).tolist()]
    sources = ", ".join(sorted(group["source"].dropna().unique()))
    summary_rows.append(
        {
            "publication": publication,
            "records": len(group),
            "sources": sources,
            "representative_titles": " | ".join(titles),
            "extractive_summary": extractive_summary(group["summary_text"].tolist()),
        }
    )

publication_summaries = pd.DataFrame(
    summary_rows,
    columns=[
        "publication",
        "records",
        "sources",
        "representative_titles",
        "extractive_summary",
    ],
)
publication_summaries

## Export Results

In [ ]:
export_df = df.copy()
if "raw_payload" in export_df:
    export_df["raw_payload"] = export_df["raw_payload"].apply(
        lambda value: json.dumps(value, ensure_ascii=False, default=str)
    )
if "authors" in export_df:
    export_df["authors"] = export_df["authors"].apply(
        lambda value: json.dumps(value, ensure_ascii=False, default=str)
    )

export_df.to_csv(RECORDS_OUTPUT, index=False)
publication_summaries.to_csv(SUMMARY_OUTPUT, index=False)

print(f"Wrote records to {RECORDS_OUTPUT}")
print(f"Wrote summaries to {SUMMARY_OUTPUT}")